# Chapter 17: Four ways to be wrong

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "17"
__file__ = os.path.abspath("run.py")

## The scene

Monday 3 August, 09:00. Varma has four words on the whiteboard and wants an example under
each, from LANTERN's own record or made on the bench.

Poisoning: one wrong entry that every later session inherits. Distraction: so much history that
the model repeats it. Confusion: tools and pages that have nothing to do with the task. Clash:
two parts of the context that disagree.

"The Pellmar line is the first," Okafor says.

"It is. I want to see the other three happen as well." Varma underlines each word. "If I have
seen it done on purpose, I will know it when it is done to us."

## The concept

A context can be wrong in ways that have nothing to do with the model's knowledge. Four are
common enough to have names:

- **Poisoning.** A false statement enters the context and is carried forward: in a memory, in
  a summary written while it was loaded, in every session after. Nobody retrieves it on
  purpose, so nobody questions it.
- **Distraction.** The context grows long with history, and the model leans on what it said
  before in place of the evidence in front of it.
- **Confusion.** The context holds things the task does not need, tools above all, and the
  model uses them.
- **Clash.** Two parts of the context disagree, and the model picks one without saying so.

A fifth question runs under all of them: how full can the window get and still be searched?
That is **long context**, and the local arm can only fill 8,192 tokens of it.

Each failure has a remedy earlier in this lab: provenance and the write gate for poisoning
(chapters 12 and 13), compaction and notes for distraction (chapters 10 and 11), less unneeded
material in the window for confusion (chapter 07), and a check in code for clash, which this
chapter writes.

```mermaid
flowchart TD
    C[LANTERN's context] --> P[Poisoning: a false entry, loaded every session]
    C --> D[Distraction: history grown towards the window]
    C --> F[Confusion: tools the task does not need]
    C --> K[Clash: an entry against a statement]
    P --> P1[Chapter 02's recorded answers, read in code]
    D --> D1[Three questions, history 0 to 6,289 tokens]
    F --> F1[Six questions, 3 tools against 19]
    K --> K1[Three seeds, memory first and statement first]
    K --> K2[clashes: the same pair found in code, no model]
    C --> L[Long context: a register grown to the window]
```

## How it works

1. **Poisoning: one entry, inherited.** No new model call: chapter 02's 26 recorded answers are
   read for the words of the four entries written under eow-fin-07 (`failures.echoes`).
2. **Distraction: a history grown towards the window.** Three questions answered from the two
   statements, after 0, 2,000, 4,000 and 6,289 tokens of LANTERN's own earlier answers
   (`failures.history`), the last as much as the window leaves room for.
3. **Confusion: tools the task does not need.** Six statement questions with the three tools
   they need, and with all 19 of LANTERN's tools, only those three connected.
4. **Clash: a memory entry against a statement.** A copy of the case memory with one entry
   added that gives 269,400.00 for the payment the statement shows as 296,400.00. The question
   is asked on three seeds with the memory before the statement and after it. Then `clashes`
   compares every entry's figures with the statements it describes, in code.
5. **Long context: the window, filled.** One account asked for in the middle of a register of
   made-up accounts grown to 1,000, 3,000, 5,000 and 7,400 tokens.

The clash check needs no model. An entry describes a statement when it names the account, its
holder or a counterparty, and its dates fall inside the statement's; a figure that is on no row
of a statement it describes is a clash:

```python
shown = {_amount(r[k]) for n in described for r in stmts[n]
         for k in ("debit", "credit", "balance") if r[k]}
for f in figures:
    if _amount(f) not in shown:
        out.append({"entry": entry, "figure": f, "statements": described})
```

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import json

from lantern import baseline, case, context, failures, learn
from lantern.config import ROOT, load_config
from lantern.llm import chat, tier
from lantern.ui import Table, chapter, console, evidence, finding, note, reply, save_shot, step

chapter("17", "Four ways to be wrong", """
Monday 3 August, 09:00. Varma has four words on the whiteboard and wants an example under
each, from LANTERN's own record or made on the bench.

Poisoning: one wrong entry that every later session inherits. Distraction: so much history that
the model repeats it. Confusion: tools and pages that have nothing to do with the task. Clash:
two parts of the context that disagree.

"The Pellmar line is the first," Okafor says.

"It is. I want to see the other three happen as well." Varma underlines each word. "If I have
seen it done on purpose, I will know it when it is done to us."
""")

MINE = learn.mine()  # --mine: your clashes() from practice/exercise.py
CLASHES = MINE.clashes if MINE else failures.clashes
OUT = ROOT / ".scratch" / "17"
OUT.mkdir(parents=True, exist_ok=True)
window = load_config()["generation"]["num_ctx"]
SEED = baseline.SEED
ORDERS = case.read("state/standing-orders.md")
STATEMENTS = "\n\n".join(f"Statement {n}:\n" + case.read(f"statements/{n}.csv")
                         for n in ("MTB-31-02290", "ICB-77-10458"))
Q = {q["id"]: q for q in baseline.questions()}
DISTRACT = ("Q-04", "Q-06", "Q-08")
CONFUSE = ("Q-01", "Q-02", "Q-03", "Q-05", "Q-06", "Q-08")
PLANTED = ("Pellmar Nominees Ltd, client account ICB-77-10458: paid 269,400.00 to Ostrander "
           "Marine Finance SA on 24 June.")
CLASH_Q = ("How much left ICB-77-10458 for Ostrander Marine Finance SA on 24 June? Give the "
           "figure and the record you took it from.")
LOCAL_SIZES = (1000, 3000, 5000, 7400)
HOSTED_SIZES = (128_000, 500_000)


def ask(system, question, *, earlier=(), seed=SEED, max_tokens=200):
    messages = [{"role": "system", "content": system}, *earlier,
                {"role": "user", "content": question}]
    return chat(messages, seed=seed, max_tokens=max_tokens)

### 1. Poisoning: one entry, inherited

In [ ]:
step("1. Poisoning: one entry, inherited")
p_poison = learn.predict_number(
    "Chapter 02 recorded LANTERN's answers to the 26 signed questions. How many of them repeat, "
    "in its own words, one of the four entries written under eow-fin-07?", unit="answers")
ref = baseline.reference(chat=chat)
reached = {r["id"]: failures.echoes(r["answer"]) for r in ref}
reached = {k: v for k, v in reached.items() if v}
table = Table(title="Chapter 02's recorded answers that repeat a false entry")
for col in ("question", "slice", "entries repeated", "passed", "the answer begins"):
    table.add_column(col)
for r in ref:
    if r["id"] in reached:
        table.add_row(r["id"], r["slice"], ", ".join(reached[r["id"]]), str(r["passed"]),
                      " ".join(r["answer"].split())[:60])
console.print(table)
by_entry = {k: sum(k in v for v in reached.values()) for k in failures.POISON}
first = learn.results("01") or {}
lines = first.get("metrics", {}).get("belief_lines")
note(f"{len(reached)} of {len(ref)} answers repeat a false entry: "
     + ", ".join(f"{k} in {n}" for k, n in by_entry.items())
     + f". {sum(not ref_r['passed'] for ref_r in ref if ref_r['id'] in reached)} of them "
     "failed. "
     + (f"Chapter 01 found the word 'cleared' on {lines} lines in "
        f"{first['metrics'].get('belief_sections')} sections: the entry and every summary "
        "written while it was loaded." if lines else "Chapter 01 has not been run here."))
learn.reveal(p_poison, actual=len(reached), tolerance=0.3,
             why="A false entry in memory is loaded into every session and written into every "
             "summary made while it was there. Nobody retrieves it on purpose, so nobody "
             "questions it.")

### 2. Distraction: a history grown towards the window

In [ ]:
step("2. Distraction: a history grown towards the window")
base_size = context.count_tokens(ORDERS + STATEMENTS)
room = window - base_size - 400
sizes = [0, *(n for n in (2000, 4000) if n < room), room]
p_dist = learn.predict(
    f"Three questions whose answers are in the statements, asked after a history of "
    f"LANTERN's own earlier answers grown to {room:,} tokens. How many does it answer right?",
    ["All three", "One or two", "None"])
dist = []
for qid in DISTRACT:
    for size in sizes:
        earlier = failures.history(ref, size, skip=qid, count=context.count_tokens)
        r = ask(ORDERS + "\n" + STATEMENTS, Q[qid]["question"], earlier=earlier)
        dist.append({"id": qid, "history": size, "turns": len(earlier) // 2,
                     "tokens": r.prompt_tokens, "answer": r.text,
                     "right": baseline.grade(r.text, Q[qid]["expect"]),
                     "copied": failures.copied(r.text, earlier),
                     "echoes": failures.echoes(r.text)})
table = Table(title="The same three questions after a longer and longer history")
for col in ("history, tokens", "earlier exchanges", "prompt tokens", *DISTRACT,
            "copied from history"):
    table.add_column(col, justify="right")
for size in sizes:
    rs = [d for d in dist if d["history"] == size]
    table.add_row(f"{size:,}", str(rs[0]["turns"]), f"{max(d['tokens'] for d in rs):,}",
                  *["right" if d["right"] else "wrong" for d in rs],
                  str(sum(d["copied"] for d in rs)))
console.print(table)
right_at = {s: sum(d["right"] for d in dist if d["history"] == s) for s in sizes}
copied_n = sum(d["copied"] for d in dist)
wrong_long = next((d for d in dist if d["history"] == room and not d["right"]), None)
if wrong_long:
    reply(f"{wrong_long['id']} after {room:,} tokens of history", wrong_long["answer"][:500],
          meta=Q[wrong_long["id"]]["question"][:90], style="red")
learn.reveal(p_dist, answer=0 if right_at[room] == 3 else (2 if right_at[room] == 0 else 1),
             why="The statements did not move. What grew was the share of the window taken by "
             "earlier answers, and a model reads its own earlier answers as a pattern to "
             "follow.")

### 3. Confusion: tools the task does not need

In [ ]:
step("3. Confusion: tools the task does not need")
all_tools = context.tools()
needed = [t for t in all_tools if t["function"]["name"] in ("statement", "sum_payments",
                                                            "money_out")]
p_conf = learn.predict_number(
    f"Six questions that need a statement. With all {len(all_tools)} of LANTERN's tools offered, "
    "only three of them connected, how many calls go to a tool that is not connected?",
    unit="calls")
conf = {"the 3 it needs": [], f"all {len(all_tools)}": []}
for qid in CONFUSE:
    for arm, schemas in zip(conf, (needed, all_tools), strict=True):
        got = baseline.answer([{"role": "system", "content": ORDERS},
                               {"role": "user", "content": Q[qid]["question"]}], schemas,
                              chat=chat, seed=SEED, max_tokens=200)
        conf[arm].append({"id": qid, **got, "right": baseline.grade(got["answer"],
                                                                    Q[qid]["expect"]),
                          "unconnected": [c for c in got["called"]
                                          if c not in ("statement", "sum_payments",
                                                       "money_out")]})
table = Table(title="The same six questions, two sets of tools")
for col in ("tools offered", "right", "calls", "calls to tools not connected", "which"):
    table.add_column(col)
for arm, rs in conf.items():
    wrong_calls = [c for r in rs for c in r["unconnected"]]
    table.add_row(arm, f"{sum(r['right'] for r in rs)} of {len(rs)}",
                  str(sum(len(r["called"]) for r in rs)), str(len(wrong_calls)),
                  ", ".join(sorted(set(wrong_calls))) or "none")
console.print(table)
arms = list(conf)
unconnected = sum(len(r["unconnected"]) for r in conf[arms[1]])
learn.reveal(p_conf, actual=unconnected, tolerance=0.5,
             why="Every schema in the request is an invitation. A tool that has nothing to do "
             "with the task still costs tokens on every request, and some of the time it is "
             "called.")

### 4. Clash: a memory entry against a statement

In [ ]:
step("4. Clash: a memory entry against a statement")
stmts, holders = failures.statements()
memory = [x[2:] for x in case.read("state/case-memory.md").splitlines() if x.startswith("- ")]
real_clash = CLASHES(memory, stmts, holders)
planted_clash = CLASHES([*memory, PLANTED], stmts, holders)
evidence("the entry added to a copy of the case memory", PLANTED)
p_clash = learn.predict(
    "The statement says 296,400.00 left for Ostrander on 24 June; the added entry says "
    "269,400.00. With the memory before the statement, as LANTERN loads it, which figure does "
    "it give?", ["The statement's", "The entry's", "Both, or neither"])
copy = case.read("state/case-memory.md").rstrip() + f"\n- {PLANTED}\n"
statement = "Statement ICB-77-10458:\n" + case.read("statements/ICB-77-10458.csv")
clash = {"memory first": [], "statement first": []}
for seed in (7, 8, 9):
    for arm in clash:
        parts = (copy, statement) if arm == "memory first" else (statement, copy)
        r = ask(ORDERS + "\n" + "\n".join(parts), CLASH_Q, seed=seed)
        stmt, entry = baseline.says(r.text, "296400.00") or baseline.says(r.text, "296400"), \
            baseline.says(r.text, "269400.00") or baseline.says(r.text, "269400")
        clash[arm].append({"seed": seed, "answer": r.text, "statement": stmt and not entry,
                           "entry": entry and not stmt, "both": stmt and entry})
table = Table(title="Which figure LANTERN gives, three seeds each")
for col in ("order", "the statement's", "the entry's", "both or neither"):
    table.add_column(col, justify="left" if col == "order" else "right")
for arm, rs in clash.items():
    table.add_row(arm, str(sum(r["statement"] for r in rs)), str(sum(r["entry"] for r in rs)),
                  str(sum(not r["statement"] and not r["entry"] for r in rs)))
console.print(table)
note(f"The clash check in code, on the real memory: {len(real_clash)} entries clash with a June "
     f"statement. On the copy: {len(planted_clash)}, "
     + "; ".join(f"{c['figure']} against {', '.join(c['statements'])}" for c in planted_clash)
     + ". It needs no model call.")
mf = clash["memory first"]
learn.reveal(p_clash, answer=0 if sum(r["statement"] for r in mf) > len(mf) / 2 else
             (1 if sum(r["entry"] for r in mf) > len(mf) / 2 else 2),
             why="When two parts of the context disagree the model picks one, and it does not "
             "say that it picked. A clash found in code can be shown to a person; a clash "
             "resolved by the model is invisible.")

### 5. Long context: the window, filled

In [ ]:
step("5. Long context: the window, filled")
p_long = learn.predict(
    "One account in the middle of a register grown to 1,000, 3,000, 5,000 and 7,400 tokens. "
    "At how many sizes does LANTERN give its account number?",
    ["All four", "Some of them", "None"])
long = []
for size in LOCAL_SIZES:
    text, question, answer = failures.haystack(size, count=context.count_tokens)
    r = chat([{"role": "user", "content": "Register of accounts:\n" + text + "\n\n" + question}],
             seed=SEED, max_tokens=40)
    long.append({"size": size, "tokens": r.prompt_tokens, "right": answer in r.text,
                 "answer": r.text[:60], "want": answer})
table = Table(title="One account, asked for in the middle of a growing register")
for col in ("register, tokens asked for", "prompt tokens", "answer", "right"):
    table.add_column(col)
for x in long:
    table.add_row(f"{x['size']:,}", f"{x['tokens']:,}", x["answer"], str(x["right"]))
console.print(table)
hosted_long = []
if tier() == "hosted":
    for size in HOSTED_SIZES:
        text, question, answer = failures.haystack(size, count=context.count_tokens)
        r = chat([{"role": "user", "content": "Register of accounts:\n" + text + "\n\n"
                   + question}], max_tokens=40)
        hosted_long.append({"size": size, "tokens": r.prompt_tokens, "right": answer in r.text})
    note("Hosted: " + ", ".join(f"{x['size']:,} asked, {x['tokens']:,} sent, right "
                                f"{x['right']}" for x in hosted_long))
else:
    note(f"The hosted arm grows the same register to {HOSTED_SIZES[0]:,} and "
         f"{HOSTED_SIZES[1]:,} tokens on {load_config()['hosted']['model']} "
         "(make lesson N=17 TIER=hosted). Not run: no hosted key.")
long_right = sum(x["right"] for x in long)
learn.reveal(p_long, answer=0 if long_right == len(long) else (2 if long_right == 0 else 1),
             why="Finding one line is the easiest thing to ask of a long context. Chapter 06 "
             "measured where in the window a fact is missed; here the question is only "
             "whether the window can be filled at all and still searched.")

(OUT / "runs.json").write_text(json.dumps({"poison": reached, "distraction": dist,
                                           "confusion": conf, "clash": clash, "long": long,
                                           "hosted_long": hosted_long}, indent=2),
                               encoding="utf-8")
learn.record(
    poison_answers=len(reached),
    poison_failed=sum(not r["passed"] for r in ref if r["id"] in reached),
    poison_m08=by_entry["M-08"],
    distraction_room=room,
    distraction_right_none=right_at[0],
    distraction_right_full=right_at[room],
    distraction_copied=copied_n,
    confusion_right_needed=sum(r["right"] for r in conf[arms[0]]),
    confusion_right_all=sum(r["right"] for r in conf[arms[1]]),
    confusion_unconnected_needed=sum(len(r["unconnected"]) for r in conf[arms[0]]),
    confusion_unconnected_all=unconnected,
    clash_real=len(real_clash),
    clash_planted=len(planted_clash),
    clash_memory_first_statement=sum(r["statement"] for r in clash["memory first"]),
    clash_memory_first_entry=sum(r["entry"] for r in clash["memory first"]),
    clash_statement_first_statement=sum(r["statement"] for r in clash["statement first"]),
    clash_statement_first_entry=sum(r["entry"] for r in clash["statement first"]),
    long_right=long_right,
    long_sizes=len(long),
    long_largest_tokens=max(x["tokens"] for x in long),
    hosted_long_run=bool(hosted_long),
    mine=MINE is not None,
)

finding(f"""
Poisoning: {len(reached)} of chapter 02's 26 recorded answers repeat an entry written under
eow-fin-07, the eighth in {by_entry['M-08']}. Distraction: three statement questions answered
{right_at[0]} of 3 with no history and {right_at[room]} of 3 after {room:,} tokens of LANTERN's
earlier answers; {copied_n} answers copied a passage from that history. Confusion: with the 3
tools it needs, {sum(r['right'] for r in conf[arms[0]])} of 6 right; with all
{len(all_tools)}, {sum(r['right'] for r in conf[arms[1]])} of 6 and {unconnected} calls to tools
that are not connected. Clash: with the memory first, LANTERN gave the statement's figure
{sum(r['statement'] for r in clash['memory first'])} times of 3 and the entry's
{sum(r['entry'] for r in clash['memory first'])}; the check in code found the planted entry and
no clash in the real memory. Long context: the account was found at {long_right} of
{len(long)} sizes up to {max(x['tokens'] for x in long):,} tokens. The Pellmar line is a
poisoning, and each of the other three has now been produced on purpose.
""")
save_shot("17-how-context-fails")

## The nuances

- **Poisoning reached five answers, and every one failed.** 5 of chapter 02's 26 recorded
  answers repeat a false entry: the eighth entry in 4, M-14 in 3, M-12 in 1, M-16 in none. Q-17
  begins "Yes. The FIU cleared Pellmar Nominees on 2 June." Chapter 01 found the word "cleared"
  on 8 lines in 6 sections: the entry and every summary written while it was loaded.
- **Distraction cost one answer, and copying cost none.** With no history the three questions
  were answered 2 of 3; after 6,289 tokens of history, 2 of 3 again, but not the same two. Q-08
  was "not in evidence" with no history and right with any. Q-04 was right until the last size,
  then gave 2,284,893.10, the balance the reserve MTB-31-02290 holds, for the balance of
  ICB-77-10458. Three answers to Q-06 copied a passage word for word from the history, and all
  three were right.
- **Confusion did not appear as planned.** With the 3 tools it needs LANTERN answered 1 of 6,
  and four of the five wrong answers called no tool at all and said "not in evidence". With
  all 19, it called `statement` for four questions and answered 5 of 6. The one call to a tool
  that is not connected, `lookup_company` for Ostrander's account number, cost that answer. On
  one seed and six questions this is thin evidence either way.
- **In the clash the entry won when it came last.** With the statement first and the memory
  after it, LANTERN gave the entry's 269,400.00 on all three seeds, and said it was "recorded in
  the statement ICB-77-10458". With the memory first it gave neither figure: 3600.00 on every
  seed, the balance after the payment, reading "how much left" as what remained.
- **A clash found in code can be shown to a person.** `clashes` found no clash between the
  sixteen real entries and the June statements, and found the planted one: 269,400.00 against
  ICB-77-10458. The model, given the same pair, chose a figure and did not say there was a
  choice.
- **The window could be filled and still searched, nearly.** The account was found at 3 of 4
  sizes up to 7,453 tokens. At 5,000 the answer was HDB-23-89593 where the register says
  MTB-23-89593: the right digits under the wrong bank.
- **The four overlap.** The poisoned answers are poisoning carried by a summary, and the Q-04
  answer is distraction that looks like a clash between two accounts. Naming the failure is
  what tells you which earlier chapter's remedy applies.

## Try this

1. Open `.scratch/17/runs.json` and read the three "statement first" clash answers. Where does
   each say its figure came from?
2. Change the clash question to "What amount was paid to Ostrander Marine Finance SA on 24
   June?" and run step 4 again. Does the memory first arm now give a figure?
3. Offer the 3 needed tools with standing order 7 quoted in the question, and count the tool
   calls.

## Master it

After this chapter you can name the four ways a context fails, produce each on purpose and
measure its effect, find a clash between memory and evidence in code, and say which earlier
remedy each failure calls for.

**Predict, then see.** `make lesson N=17` stops five times: how many recorded answers repeat a
false entry, how many questions survive a full history, how many calls go to tools that are not
connected, which figure LANTERN gives in the clash, and at how many sizes it finds the account.

**Build it.** Write `clashes` in [`practice/exercise.py`](practice/exercise.py):

```bash
make exercise N=17          # 7 offline tests
make hint N=17
make lesson N=17 ARGS=--mine
```

**Check yourself.** `make quiz N=17`: six questions.

**Break it.** `make challenge N=17`: your check on the real memory and the copy; a clash
question that the memory first arm answers with a figure; and a clash check that misses the
planted entry.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 17

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 17

In [ ]:
%run ../../scripts/learn.py hint 17

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 17

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

The Pellmar line is poisoning: 5 of chapter 02's 26 recorded answers repeat an entry written
under eow-fin-07, the eighth in 4, and all 5 failed. Distraction was produced at the edge of the
window: after 6,289 tokens of history one answer took the reserve's balance for Pellmar's.
Confusion was not produced as planned: with the 3 tools it needs LANTERN answered 1 of 6, with
all 19, 5 of 6 and one call to a tool that is not connected. The clash was: with the entry
after the statement LANTERN gave the entry's figure on every seed and credited it to the
statement, and the check in code found the planted entry and nothing in the real memory. The
window filled to 7,453 tokens could still be searched at 3 of 4 sizes. Varma has an example
under each word, and the one that matters most is still the first.

**Next:** [Chapter 18: Retraction](../18-correcting-and-forgetting/README.md)